# Analisis Perbandingan Algoritma Random Forest dan XGBoost Teroptimasi untuk Klasifikasi Risiko Penyakit Jantung

**Topik Penelitian:** Klasifikasi Risiko Penyakit Jantung (Heart Disease Classification)
**Dataset:** UCI Cleveland Heart Disease Dataset (303 sampel, 14 atribut)
**Tujuan:** Membandingkan performa model klasifikasi Random Forest dan XGBoost sebelum dan sesudah optimasi hyperparameter menggunakan Grid Search CV.

## 1. Import Library & Persiapan Lingkungan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

sns.set_theme(style="whitegrid")
%matplotlib inline

## 2. Memuat dan Memeriksa Dataset

In [ ]:
df = pd.read_csv('../data/heart_disease.csv')
print(f"Bentuk Dataset: {df.shape}")
df.head()

In [ ]:
df.info()
print("\nDistribusi Target:")
print(df['target'].value_counts())

## 3. Pra-pemrosesan Data & Feature Scaling

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

# Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling Fitur
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

print(f"Ukuran X_train: {X_train_scaled.shape}, Ukuran X_test: {X_test_scaled.shape}")

## 4. Pemodelan & Optimasi Hyperparameter (GridSearchCV)

In [ ]:
# 1. Baseline Random Forest
rf_base = RandomForestClassifier(random_state=42)
rf_base.fit(X_train_scaled, y_train)

# 2. Baseline XGBoost
xgb_base = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_base.fit(X_train_scaled, y_train)

# 3. Grid Search RF
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10]
}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_train_scaled, y_train)
rf_opt = grid_rf.best_estimator_

# 4. Grid Search XGB
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}
grid_xgb = GridSearchCV(XGBClassifier(random_state=42, eval_metric='logloss'), param_grid_xgb, cv=5, scoring='accuracy', n_jobs=-1)
grid_xgb.fit(X_train_scaled, y_train)
xgb_opt = grid_xgb.best_estimator_

print("Pelatihan dan Optimasi Selesai!")

## 5. Evaluasi & Perbandingan Hasil

In [ ]:
models = {
    'Random Forest (Baseline)': rf_base,
    'XGBoost (Baseline)': xgb_base,
    'Random Forest (Optimized)': rf_opt,
    'XGBoost (Optimized)': xgb_opt
}

results = []
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 4)
    })

df_res = pd.DataFrame(results)
df_res

## 6. Visualisasi Feature Importance

In [ ]:
importances = xgb_opt.feature_importances_
df_imp = pd.DataFrame({'Feature': X.columns, 'Importance': importances}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_imp, x='Importance', y='Feature', palette='viridis')
plt.title('Fitur Paling Dominan dalam Prediksi Risiko Penyakit Jantung (XGBoost Optimized)', fontsize=14)
plt.xlabel('Nilai Importance')
plt.ylabel('Fitur Medis')
plt.tight_layout()
plt.show()